## User Story

As an ML Engineer, I want to investigate failure cases when using
LLM-as-a-Judge for English-to-Mandarin translation evaluation so that
we can understand where automated LLM-based evaluation becomes
unreliable or inconsistent.

## Objectives

This experiment investigates situations in which an LLM judge may:

- produce inconsistent judgments across repeated evaluations,
- miss important translation errors,
- show bias toward superficial translation characteristics,
- change its judgment when the evaluation setup changes,
- or fail to follow the required evaluation format.

The study focuses on English-to-Mandarin translation evaluation.

## Failure Categories

The following categories will be investigated:

1. Run-to-run inconsistency
2. Prompt sensitivity
3. Semantic-error blindness
4. Fluency bias
5. Reference sensitivity
6. Evaluation-setup sensitivity

Repeated evaluations will be used where appropriate to distinguish
systematic behavior from stochastic variation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import json
import random
import torch

from collections import Counter
from tqdm.auto import tqdm

seed=42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print("pytorch:", torch.__version__)
print("cuda_available:",torch.cuda.is_available())

pytorch: 2.11.0+cu130
cuda_available: True


In [2]:
#config setting

EXPERIMENT_CONFIG={
    "judge_model":"Qwen/Qwen3-4B",
    "temperature":0.7,
    "top_p":0.95,
    "repeat_runs":5,
    "seed":seed
}

print("Failure case config:")
print("-"*20)

for key,value in EXPERIMENT_CONFIG.items():
  print(f"{key}:{value}")

Failure case config:
--------------------
judge_model:Qwen/Qwen3-4B
temperature:0.7
top_p:0.95
repeat_runs:5
seed:42


## Experimental Principle

Failure cases are investigated using controlled comparisons.

Whenever possible, only one property of a translation or evaluation
setup is changed at a time. This helps determine whether a change in
the judge's output is associated with the tested factor rather than
with unrelated differences between examples.

Each relevant condition is evaluated repeatedly because a single LLM
response cannot distinguish systematic judge behavior from stochastic
generation variability.

The judge configuration is kept consistent with the previous
prompt-engineering experiment:

- Judge: Qwen3-4B
- Temperature: 0.7
- Top-p: 0.95
- Repeated runs: 5

# Failure-Case Experimental Design

This study uses two complementary forms of evidence.

## Part A — Naturally Observed Failure Cases

Outputs from the previous WMT17 prompt-engineering experiment are analyzed
to identify naturally occurring judge failures, including:

- run-to-run score inconsistency,
- sensitivity to prompt structure,
- unusually large score disagreements,
- and output-format/instruction-following problems.

These cases use real WMT17 English-to-Mandarin examples and fixed
Helsinki-NLP candidate translations.

## Part B — Controlled Failure Cases

Selected English-to-Mandarin examples from WMT17 are modified in controlled
ways to introduce known translation errors.

Only one semantic property is changed whenever possible. This allows us to
test whether the judge recognizes specific translation errors.

Target failure categories include:

- numerical errors,
- negation errors,
- named-entity errors,
- omissions,
- unsupported additions,
- temporal errors,
- modality errors,
- terminology errors.

## Why Both Are Needed

Natural cases show how the judge behaves on realistic machine translations.

Controlled cases provide known error conditions that make it possible to
determine whether the judge actually detects important translation errors.

Consistency and correctness are analyzed separately because a judge may be
highly consistent while repeatedly producing an incorrect assessment.

In [3]:
task1_results = pd.read_csv("judge_full_results.csv")
eval_df = pd.read_csv("wmt17_en_zh_judge_eval_50.csv")

print("Task 1 judge results:", task1_results.shape)
print("Evaluation examples:", eval_df.shape)

print("\nTask 1 result columns:")
print(task1_results.columns.tolist())

print("\nEvaluation dataset columns:")
print(eval_df.columns.tolist())

Task 1 judge results: (1000, 11)
Evaluation examples: (50, 4)

Task 1 result columns:
['eval_id', 'prompt', 'run', 'score', 'accuracy', 'completeness', 'fluency', 'terminology', 'valid_output', 'explanation', 'raw_response']

Evaluation dataset columns:
['eval_id', 'source_en', 'reference_zh', 'candidate_zh']


In [4]:
assert len(task1_results) == 1000,(
    f"length of task 1 results shold be 1000,but length is {len(task1_results)}"
)

assert len(eval_df) == 50,(
    f"length of evaluation dataset should be 50,but length is {len(eval_df)}"
)

expected_prompts = {
    "P0_baseline",
    "P1_rubric",
    "P2_anchored",
    "P3_decomposed"
}

actual_prompt= set(task1_results['prompt'].unique())

assert actual_prompt == actual_prompt ,(
    f"unexpected prompt config: {actual_prompt}"
)


In [5]:
run_consistency = (
    task1_results
    .groupby(["eval_id", "prompt"])["score"]
    .agg(
        mean_score="mean",
        std_score="std",
        min_score="min",
        max_score="max",
        unique_scores="nunique",
        n_runs="count"
    )
    .reset_index()
)

run_consistency["score_range"] = (
    run_consistency["max_score"]
    - run_consistency["min_score"]
)

run_consistency["inconsistent"] = (
    run_consistency["unique_scores"] > 1
)

print("Prompt-example combinations:", len(run_consistency))
print(
    "Inconsistent combinations:",
    run_consistency["inconsistent"].sum()
)

run_consistency.head()

Prompt-example combinations: 200
Inconsistent combinations: 21


,eval_id,prompt,mean_score,std_score,min_score,max_score,unique_scores,n_runs,score_range,inconsistent
0,E001,P0_baseline,4.0,0.0,4,4,1,5,0,False
1,E001,P1_rubric,4.0,0.0,4,4,1,5,0,False
2,E001,P2_anchored,4.0,0.0,4,4,1,5,0,False
3,E001,P3_decomposed,5.0,0.0,5,5,1,5,0,False
4,E002,P0_baseline,3.0,0.0,3,3,1,5,0,False


In [6]:
worst_inconsistency = (
    run_consistency[
        run_consistency["inconsistent"]
    ]
    .sort_values(
        ["score_range", "std_score"],
        ascending=False
    )
)

worst_inconsistency.head(15)

,eval_id,prompt,mean_score,std_score,min_score,max_score,unique_scores,n_runs,score_range,inconsistent
7,E002,P3_decomposed,1.6,0.547723,1,2,2,5,1,True
91,E023,P3_decomposed,3.6,0.547723,3,4,2,5,1,True
159,E040,P3_decomposed,3.4,0.547723,3,4,2,5,1,True
196,E050,P0_baseline,3.6,0.547723,3,4,2,5,1,True
32,E009,P0_baseline,3.6,0.547723,3,4,2,5,1,True
105,E027,P1_rubric,3.4,0.547723,3,4,2,5,1,True
172,E044,P0_baseline,3.4,0.547723,3,4,2,5,1,True
197,E050,P1_rubric,3.4,0.547723,3,4,2,5,1,True
113,E029,P1_rubric,4.4,0.547723,4,5,2,5,1,True
82,E021,P2_anchored,4.2,0.447214,4,5,2,5,1,True


In [7]:
worst_inconsistency = (
    run_consistency[
        run_consistency["inconsistent"]
    ]
    .sort_values(
        ["score_range", "std_score"],
        ascending=False
    )
)

worst_inconsistency.head(15)

,eval_id,prompt,mean_score,std_score,min_score,max_score,unique_scores,n_runs,score_range,inconsistent
7,E002,P3_decomposed,1.6,0.547723,1,2,2,5,1,True
91,E023,P3_decomposed,3.6,0.547723,3,4,2,5,1,True
159,E040,P3_decomposed,3.4,0.547723,3,4,2,5,1,True
196,E050,P0_baseline,3.6,0.547723,3,4,2,5,1,True
32,E009,P0_baseline,3.6,0.547723,3,4,2,5,1,True
105,E027,P1_rubric,3.4,0.547723,3,4,2,5,1,True
172,E044,P0_baseline,3.4,0.547723,3,4,2,5,1,True
197,E050,P1_rubric,3.4,0.547723,3,4,2,5,1,True
113,E029,P1_rubric,4.4,0.547723,4,5,2,5,1,True
82,E021,P2_anchored,4.2,0.447214,4,5,2,5,1,True


In [8]:
natural_inconsistency_cases = worst_inconsistency.merge(
    eval_df[
        [
            "eval_id",
            "source_en",
            "reference_zh",
            "candidate_zh"
        ]
    ],
    on="eval_id",
    how="left"
)

natural_inconsistency_cases[
    [
        "eval_id",
        "prompt",
        "source_en",
        "reference_zh",
        "candidate_zh",
        "min_score",
        "max_score",
        "score_range",
        "std_score"
    ]
].head(10)

,eval_id,prompt,source_en,reference_zh,candidate_zh,min_score,max_score,score_range,std_score
0,E002,P3_decomposed,"Russian gymnast Aliya Mustafina, with her larg...",长着一双会说话的大眼睛的俄罗斯体操选手穆斯塔芬娜 (Aliya Mustafina) 被指长...,"俄罗斯体育馆Aliya Mustafina, 以她大眼睛的表情, 据说是古巴裔美国人歌手Gl...",1,2,1,0.547723
1,E023,P3_decomposed,Great Britain's impressive run in the Rugby 7s...,在周四晚上的七人制橄榄球赛决赛中，英国队未能在斐济队面前延续之前的出色表现，被斐济队以43比...,"斐济在周四晚上的决赛中在最后障碍处陷入了崩溃, 当时他们在里约的Deodoro体育场43-7...",3,4,1,0.547723
2,E040,P3_decomposed,But the reality is that gymnastics is an expen...,但是现实却是，体操是一项费用昂贵的运动，体操运动员主要在私人体操俱乐部进行训练，她们的父母需...,"但实际情况是,体操是一项昂贵的运动,体操运动员主要是私人体操俱乐部的训练,由父母负责支付培训...",3,4,1,0.547723
3,E050,P0_baseline,The consolidated payment institution shall com...,被合并支付机构应于公告之日起6个月内完成支付业务承接工作。,综合付款机构应在通知之日起6个月内完成付款服务。,3,4,1,0.547723
4,E009,P0_baseline,"""Tourists that way, tomatoes back here,"" as a ...",莫斯科报纸 Vedemosti 的一篇报道称：“游客去了，番茄回来了。”,"维德莫斯蒂报纸的一篇报导写道,",3,4,1,0.547723
5,E027,P1_rubric,The cooperation between China and Russia is si...,作为分布在中俄交界区域的世界野生动物保护旗舰物种，中俄两国间的合作对于野生东北虎、豹的种群恢...,中国和俄罗斯之间的合作对中俄边境地区野生动物保护旗舰物种 -- -- 中俄虎和东北豹 -- ...,3,4,1,0.547723
6,E044,P0_baseline,"In July 2015, on Anzhen Street of Wuxi Xishan ...",2015年7月，无锡市锡山区安镇街道14个农村经济合作组织与所在村村委会“分账管理”，村委会...,"2015年7月,14个农村经济合作组织与村委员会合作,在武西西山县安盛街实施“子账户管理”,...",3,4,1,0.547723
7,E050,P1_rubric,The consolidated payment institution shall com...,被合并支付机构应于公告之日起6个月内完成支付业务承接工作。,综合付款机构应在通知之日起6个月内完成付款服务。,3,4,1,0.547723
8,E029,P1_rubric,According to the recent report of the Ministry...,根据民政部最近发布的情况，全国6000多万人受灾，直接经济损失超过2000亿元。,"根据民政部最近的报告,6 000多万人受到影响,直接经济损失超过2 000亿元。",4,5,1,0.547723
9,E021,P2_anchored,"As of June 2016, the world conducted a total o...",截至2016年6月，全球共进行了114次月球探测，43次火星探测。,"截至2016年6月,全世界共进行了114次月球勘探,包括43次火星探测任务。",4,5,1,0.447214


In [9]:
def show_judge_runs(eval_id, prompt_name):

    example = eval_df[
        eval_df["eval_id"] == eval_id
    ].iloc[0]

    runs = task1_results[
        (task1_results["eval_id"] == eval_id) &
        (task1_results["prompt"] == prompt_name)
    ].sort_values("run")

    print("=" * 100)
    print("Example:", eval_id)
    print("Prompt:", prompt_name)

    print("\nEnglish:")
    print(example["source_en"])

    print("\nReference:")
    print(example["reference_zh"])

    print("\nCandidate:")
    print(example["candidate_zh"])

    print("\nJudge runs:")
    print("-" * 100)

    for _, row in runs.iterrows():
        print(
            f"Run {int(row['run'])}: "
            f"Score = {row['score']}"
        )

        if "explanation" in row and pd.notna(row["explanation"]):
            print("Explanation:", row["explanation"])

        print()

In [10]:
worst_case = worst_inconsistency.iloc[0]

show_judge_runs(
    worst_case["eval_id"],
    worst_case["prompt"]
)

Example: E002
Prompt: P3_decomposed

English:
Russian gymnast Aliya Mustafina, with her large, expressive eyes, has been said to be a dead ringer for Cuban-American singer Gloria Estefan.

Reference:
长着一双会说话的大眼睛的俄罗斯体操选手穆斯塔芬娜 (Aliya Mustafina) 被指长相酷似古巴裔美国歌星格洛丽亚·埃斯特凡 (Gloria Estefan)。

Candidate:
俄罗斯体育馆Aliya Mustafina, 以她大眼睛的表情, 据说是古巴裔美国人歌手Gloria Estefan的死铃声。

Judge runs:
----------------------------------------------------------------------------------------------------
Run 1: Score = 1
Explanation: The candidate translation is highly inaccurate and contains numerous errors. It misplaces the noun '俄罗斯体育馆' (Russian gym) as an adjective describing Aliya Mustafina, which is grammatically incorrect. The phrase '以她大眼睛的表情' (with her big eyes' expression) is awkward and does not convey the intended meaning of 'large, expressive eyes.' The term '死铃声' (dead ringer) is a literal mistranslation; the correct term is '酷似' (resembles). The translation is also poorly structured and lacks fluency. Over

In [11]:
prompt_means = (
    task1_results
    .groupby(["eval_id", "prompt"])["score"]
    .mean()
    .reset_index()
)

prompt_score_matrix = (
    prompt_means
    .pivot(
        index="eval_id",
        columns="prompt",
        values="score"
    )
    .reset_index()
)

prompt_score_matrix.head()

prompt,eval_id,P0_baseline,P1_rubric,P2_anchored,P3_decomposed
0,E001,4.0,4.0,4.0,5.0
1,E002,3.0,3.0,3.0,1.6
2,E003,4.0,4.0,4.0,4.0
3,E004,4.0,4.0,4.0,5.0
4,E005,3.8,3.2,4.0,3.0


In [12]:
prompt_columns = [
    "P0_baseline",
    "P1_rubric",
    "P2_anchored",
    "P3_decomposed"
]

prompt_score_matrix["min_prompt_score"] = (
    prompt_score_matrix[prompt_columns].min(axis=1)
)

prompt_score_matrix["max_prompt_score"] = (
    prompt_score_matrix[prompt_columns].max(axis=1)
)

prompt_score_matrix["prompt_range"] = (
    prompt_score_matrix["max_prompt_score"]
    - prompt_score_matrix["min_prompt_score"]
)

prompt_score_matrix["prompt_std"] = (
    prompt_score_matrix[prompt_columns].std(axis=1)
)

prompt_score_matrix["prompt_sensitive"] = (
    prompt_score_matrix["prompt_range"] >= 1.0
)

prompt_score_matrix.sort_values(
    ["prompt_range", "prompt_std"],
    ascending=False
).head(15)

prompt,eval_id,P0_baseline,P1_rubric,P2_anchored,P3_decomposed,min_prompt_score,max_prompt_score,prompt_range,prompt_std,prompt_sensitive
1,E002,3.0,3.0,3.0,1.6,1.6,3.0,1.4,0.700000,True
27,E028,3.0,3.0,3.0,1.8,1.8,3.0,1.2,0.600000,True
13,E014,4.0,4.0,3.0,3.0,3.0,4.0,1.0,0.577350,True
20,E021,4.0,5.0,4.2,5.0,4.0,5.0,1.0,0.525991,True
0,E001,4.0,4.0,4.0,5.0,4.0,5.0,1.0,0.500000,True
3,E004,4.0,4.0,4.0,5.0,4.0,5.0,1.0,0.500000,True
7,E008,4.0,4.0,4.0,5.0,4.0,5.0,1.0,0.500000,True
9,E010,4.0,4.0,4.0,5.0,4.0,5.0,1.0,0.500000,True
10,E011,4.0,4.0,4.0,5.0,4.0,5.0,1.0,0.500000,True
14,E015,4.0,4.0,4.0,5.0,4.0,5.0,1.0,0.500000,True


In [13]:
n_prompt_sensitive = (
    prompt_score_matrix["prompt_sensitive"].sum()
)

print("Total examples:", len(prompt_score_matrix))
print(
    "Prompt-sensitive examples (range >= 1.0):",
    n_prompt_sensitive
)

print(
    "Percentage:",
    round(
        100 * n_prompt_sensitive / len(prompt_score_matrix),
        1
    ),
    "%"
)

print(
    "Average prompt score range:",
    round(prompt_score_matrix["prompt_range"].mean(), 3)
)

print(
    "Maximum prompt score range:",
    prompt_score_matrix["prompt_range"].max()
)

Total examples: 50
Prompt-sensitive examples (range >= 1.0): 28
Percentage: 56.0 %
Average prompt score range: 0.652
Maximum prompt score range: 1.4


In [14]:
prompt_sensitivity_cases = (
    prompt_score_matrix
    .sort_values(
        ["prompt_range", "prompt_std"],
        ascending=False
    )
    .merge(
        eval_df[
            [
                "eval_id",
                "source_en",
                "reference_zh",
                "candidate_zh"
            ]
        ],
        on="eval_id",
        how="left"
    )
)

prompt_sensitivity_cases[
    [
        "eval_id",
        "source_en",
        "candidate_zh",
        "P0_baseline",
        "P1_rubric",
        "P2_anchored",
        "P3_decomposed",
        "prompt_range"
    ]
].head(10)

,eval_id,source_en,candidate_zh,P0_baseline,P1_rubric,P2_anchored,P3_decomposed,prompt_range
0,E002,"Russian gymnast Aliya Mustafina, with her larg...","俄罗斯体育馆Aliya Mustafina, 以她大眼睛的表情, 据说是古巴裔美国人歌手Gl...",3.0,3.0,3.0,1.6,1.4
1,E028,But when he was defeated by his counterpart Hu...,"但是当他在2004年奥林匹克运动会的十米比赛中被对手胡佳击败时,天梁的竞争对手国家滑倒,据说...",3.0,3.0,3.0,1.8,1.2
2,E014,"With shares quantifying to each villager, the ...",农村集体经济改革的重点是明确农村集体管理和探索政治和经济分离的财产权。,4.0,4.0,3.0,3.0,1.0
3,E021,"As of June 2016, the world conducted a total o...","截至2016年6月,全世界共进行了114次月球勘探,包括43次火星探测任务。",4.0,5.0,4.2,5.0,1.0
4,E001,Democratic members of the House Intelligence C...,"众议院情报委员会的民主成员在星期四公布了他们自己的调查结果,他们同意共和党工作队的一些结论。",4.0,4.0,4.0,5.0,1.0
5,E004,"Over the past decade, the State Council has is...","过去十年来,国务院共发出五份消除过度和落后生产能力的通知。",4.0,4.0,4.0,5.0,1.0
6,E008,Even though she learns and works in decentrali...,"尽管她以分散、模块化的方式学习和工作,但她基本上能够完成任务,并取得良好成果。",4.0,4.0,4.0,5.0,1.0
7,E010,According to the forecast of Ministry of Trade...,"根据贸易和经济发展部的预测,该国的经济增长将达到7%,而该国的通货膨胀率基本上高于这一水平。",4.0,4.0,4.0,5.0,1.0
8,E011,For Ankara the benefits of calling a truce are...,"对安卡拉来说,要求休战的好处是显而易见的。",4.0,4.0,4.0,5.0,1.0
9,E015,The idiot didn't realize they were still on th...,那个白痴不知道他们还在直播,4.0,4.0,4.0,5.0,1.0


In [15]:
def show_prompt_sensitivity(eval_id):

    example = eval_df[
        eval_df["eval_id"] == eval_id
    ].iloc[0]

    scores = prompt_score_matrix[
        prompt_score_matrix["eval_id"] == eval_id
    ].iloc[0]

    print("=" * 100)
    print("Example:", eval_id)

    print("\nEnglish:")
    print(example["source_en"])

    print("\nReference:")
    print(example["reference_zh"])

    print("\nCandidate:")
    print(example["candidate_zh"])

    print("\nMean judge scores across 5 runs:")
    print("-" * 50)

    for prompt in prompt_columns:
        print(
            f"{prompt:20s}: "
            f"{scores[prompt]:.2f}"
        )

    print("\nPrompt score range:", round(scores["prompt_range"], 2))

In [16]:
most_sensitive_id = (
    prompt_sensitivity_cases.iloc[0]["eval_id"]
)

show_prompt_sensitivity(most_sensitive_id)

Example: E002

English:
Russian gymnast Aliya Mustafina, with her large, expressive eyes, has been said to be a dead ringer for Cuban-American singer Gloria Estefan.

Reference:
长着一双会说话的大眼睛的俄罗斯体操选手穆斯塔芬娜 (Aliya Mustafina) 被指长相酷似古巴裔美国歌星格洛丽亚·埃斯特凡 (Gloria Estefan)。

Candidate:
俄罗斯体育馆Aliya Mustafina, 以她大眼睛的表情, 据说是古巴裔美国人歌手Gloria Estefan的死铃声。

Mean judge scores across 5 runs:
--------------------------------------------------
P0_baseline         : 3.00
P1_rubric           : 3.00
P2_anchored         : 3.00
P3_decomposed       : 1.60

Prompt score range: 1.4


In [17]:
from itertools import combinations

pairwise_prompt_differences = []

for prompt_a, prompt_b in combinations(prompt_columns, 2):

    differences = (
        prompt_score_matrix[prompt_a]
        - prompt_score_matrix[prompt_b]
    )

    pairwise_prompt_differences.append({
        "prompt_a": prompt_a,
        "prompt_b": prompt_b,
        "mean_signed_difference": differences.mean(),
        "mean_absolute_difference": differences.abs().mean(),
        "max_absolute_difference": differences.abs().max()
    })

pairwise_prompt_df = pd.DataFrame(
    pairwise_prompt_differences
)

pairwise_prompt_df

,prompt_a,prompt_b,mean_signed_difference,mean_absolute_difference,max_absolute_difference
0,P0_baseline,P1_rubric,0.056,0.160,1.0
1,P0_baseline,P2_anchored,0.044,0.116,1.0
2,P0_baseline,P3_decomposed,-0.064,0.632,1.4
3,P1_rubric,P2_anchored,-0.012,0.108,1.0
4,P1_rubric,P3_decomposed,-0.120,0.488,1.4
5,P2_anchored,P3_decomposed,-0.108,0.540,1.4


# Controlled Semantic-Error Analysis

Natural WMT17 cases demonstrate that judge outputs can vary across repeated
runs and prompt configurations. However, disagreement alone does not establish
which judgment is correct.

To investigate judgment accuracy, a controlled perturbation experiment is
constructed from real WMT17 English-to-Mandarin examples.

For each selected example:

1. The original Mandarin reference is treated as the clean translation.
2. A controlled semantic error is introduced into a copy of the translation.
3. The clean and perturbed translations are evaluated independently.
4. The judge is expected to assign the clean translation a higher score.

Only one semantic property is modified whenever possible. This provides a
known directional expectation without requiring a human numerical quality
score.

Target error categories include:

- numerical errors,
- named-entity errors,
- negation errors,
- temporal errors,
- omission errors,
- unsupported additions,
- modality errors,
- terminology errors.

In [18]:
import re

def detect_case_features(text):

    text_lower = str(text).lower()

    return {
        "has_number": bool(
            re.search(r"\d+", text)
        ),

        "has_negation": bool(
            re.search(
                r"\b(no|not|never|without|cannot|can't|didn't|doesn't|isn't|wasn't)\b",
                text_lower
            )
        ),

        "has_modality": bool(
            re.search(
                r"\b(must|may|might|should|could|can|will|would)\b",
                text_lower
            )
        ),

        "has_temporal": bool(
            re.search(
                r"\b(today|tomorrow|yesterday|monday|tuesday|wednesday|"
                r"thursday|friday|saturday|sunday|january|february|march|"
                r"april|may|june|july|august|september|october|november|"
                r"december|year|month|week)\b",
                text_lower
            )
        )
    }


feature_rows = []

for _, row in eval_df.iterrows():

    features = detect_case_features(
        row["source_en"]
    )

    features["eval_id"] = row["eval_id"]
    features["source_en"] = row["source_en"]

    feature_rows.append(features)


feature_df = pd.DataFrame(feature_rows)

feature_df.head()

,has_number,has_negation,has_modality,has_temporal,eval_id,source_en
0,False,False,False,True,E001,Democratic members of the House Intelligence C...
1,False,False,False,False,E002,"Russian gymnast Aliya Mustafina, with her larg..."
2,True,False,False,False,E003,Gee and Schwartzman are scheduled to discuss a...
3,False,False,False,False,E004,"Over the past decade, the State Council has is..."
4,False,False,False,False,E005,Helen Glover and Heather Stanning inspired to ...


In [19]:
feature_summary = {
    "number": int(feature_df["has_number"].sum()),
    "negation": int(feature_df["has_negation"].sum()),
    "modality": int(feature_df["has_modality"].sum()),
    "temporal": int(feature_df["has_temporal"].sum())
}

print("Potential controlled cases")
print("-" * 40)

for category, count in feature_summary.items():
    print(f"{category:12s}: {count}")

Potential controlled cases
----------------------------------------
number      : 14
negation    : 8
modality    : 10
temporal    : 10


In [20]:
def show_feature_examples(feature_name, n=10):

    subset = feature_df[
        feature_df[feature_name]
    ][
        ["eval_id", "source_en"]
    ]

    print(
        f"{feature_name}: {len(subset)} potential examples"
    )

    return subset.head(n)


show_feature_examples("has_number", 10)

has_number: 14 potential examples


,eval_id,source_en
2,E003,Gee and Schwartzman are scheduled to discuss a...
5,E006,12-0 and GB had hardly touched the ball.
9,E010,According to the forecast of Ministry of Trade...
12,E013,"As never before or since, home economists - am..."
16,E017,"Maybe you'll get a call at midnight. ""Hello, S..."
20,E021,"As of June 2016, the world conducted a total o..."
22,E023,Great Britain's impressive run in the Rugby 7s...
23,E024,"Prior to this, Jining Municipal Committee for ..."
27,E028,But when he was defeated by his counterpart Hu...
28,E029,According to the recent report of the Ministry...


In [21]:
show_feature_examples("has_negation", 10)

has_negation: 8 potential examples


,eval_id,source_en
12,E013,"As never before or since, home economists - am..."
14,E015,The idiot didn't realize they were still on th...
30,E031,Zhang Xiaolong showed the animation show of he...
32,E033,"With the popularity of smart phones, mobile ph..."
33,E034,Parker had a better performance than last seas...
37,E038,"Last year, officials said, Mr. Hooker's team c..."
42,E043,"We are witnessing the top of the sport, but it..."
46,E047,"""I thought my elder brother and I cannot affor..."


In [22]:
show_feature_examples("has_modality", 10)

has_modality: 10 potential examples


,eval_id,source_en
6,E007,"According to the navigation needs, the old Wan..."
7,E008,Even though she learns and works in decentrali...
9,E010,According to the forecast of Ministry of Trade...
18,E019,"Their new, collaborative work, ""A Square Meal,..."
23,E024,"Prior to this, Jining Municipal Committee for ..."
25,E026,"We can bring those brands back up again, so I ..."
29,E030,He had told BMA members the deal was a good on...
30,E031,Zhang Xiaolong showed the animation show of he...
38,E039,Whether it is because of the increase in age o...
47,E048,"""We hope that by comparing our results to othe..."


In [23]:
show_feature_examples("has_temporal", 10)

has_temporal: 10 potential examples


,eval_id,source_en
0,E001,Democratic members of the House Intelligence C...
15,E016,This is despite a promise from Jeremy Hunt jus...
18,E019,"Their new, collaborative work, ""A Square Meal,..."
20,E021,"As of June 2016, the world conducted a total o..."
22,E023,Great Britain's impressive run in the Rugby 7s...
23,E024,"Prior to this, Jining Municipal Committee for ..."
37,E038,"Last year, officials said, Mr. Hooker's team c..."
40,E041,Three years ago he initially struggled when re...
41,E042,The result is mean climate temperatures that a...
43,E044,"In July 2015, on Anzhen Street of Wuxi Xishan ..."


In [24]:
candidate_cases = feature_df.merge(
    eval_df[
        [
            "eval_id",
            "reference_zh"
        ]
    ],
    on="eval_id",
    how="left"
)

candidate_cases[
    [
        "eval_id",
        "source_en",
        "reference_zh",
        "has_number",
        "has_negation",
        "has_modality",
        "has_temporal"
    ]
]

,eval_id,source_en,reference_zh,has_number,has_negation,has_modality,has_temporal
0,E001,Democratic members of the House Intelligence C...,众议院情报委员会的民主党成员于周四公布了其调查结果，与共和党专项工作小组的部分结论相一致。,False,False,False,True
1,E002,"Russian gymnast Aliya Mustafina, with her larg...",长着一双会说话的大眼睛的俄罗斯体操选手穆斯塔芬娜 (Aliya Mustafina) 被指长...,False,False,False,False
2,E003,Gee and Schwartzman are scheduled to discuss a...,吉和斯瓦茨曼将讨论并分享1926年贝特伦·古德西的设计图片以及1933年哈迪・霍尔兹曼・弗夫...,True,False,False,False
3,E004,"Over the past decade, the State Council has is...",近十年来，国务院前后共发布过5次淘汰过剩与落后产能的通知。,False,False,False,False
4,E005,Helen Glover and Heather Stanning inspired to ...,海伦·格拉芙和希瑟·斯坦宁受身患癌症的教练的鼓舞，夺得赛艇金牌,False,False,False,False
5,E006,12-0 and GB had hardly touched the ball.,12比0，英国队几乎没有碰到过球。,True,False,False,False
6,E007,"According to the navigation needs, the old Wan...","根据通航需要,老万福河航道采取养护改造的方式,对局部水深进行了浚深,局部急弯进行裁弯取直等,...",False,False,True,False
7,E008,Even though she learns and works in decentrali...,即便通过分散的，模块化的时间来学习和工作，她也基本上都可以完成任务，并取得不错的成绩。,False,False,True,False
8,E009,"""Tourists that way, tomatoes back here,"" as a ...",莫斯科报纸 Vedemosti 的一篇报道称：“游客去了，番茄回来了。”,False,False,False,False
9,E010,According to the forecast of Ministry of Trade...,根据塔贸易和经济发展部的预测，2016年塔国的经济增长将达到7%，而国家的通胀也基本是在这一...,True,False,True,False


In [25]:
number_cases = candidate_cases[
    candidate_cases["has_number"]
][
    [
        "eval_id",
        "source_en",
        "reference_zh"
    ]
]

number_cases

,eval_id,source_en,reference_zh
2,E003,Gee and Schwartzman are scheduled to discuss a...,吉和斯瓦茨曼将讨论并分享1926年贝特伦·古德西的设计图片以及1933年哈迪・霍尔兹曼・弗夫...
5,E006,12-0 and GB had hardly touched the ball.,12比0，英国队几乎没有碰到过球。
9,E010,According to the forecast of Ministry of Trade...,根据塔贸易和经济发展部的预测，2016年塔国的经济增长将达到7%，而国家的通胀也基本是在这一...
12,E013,"As never before or since, home economists - am...",家庭经济学家们影响着这个国家的饮食习惯，比如1923年至1943年担任联邦家庭经济局主厨的路...
16,E017,"Maybe you'll get a call at midnight. ""Hello, S...",也许，你半夜就会接到电话，“您好，苏宁易购，您的iPhone7到了。”
20,E021,"As of June 2016, the world conducted a total o...",截至2016年6月，全球共进行了114次月球探测，43次火星探测。
22,E023,Great Britain's impressive run in the Rugby 7s...,在周四晚上的七人制橄榄球赛决赛中，英国队未能在斐济队面前延续之前的出色表现，被斐济队以43比...
23,E024,"Prior to this, Jining Municipal Committee for ...",此前，济宁市老龄委、市财政局、市交通局下发《关于60—64周岁老年人半价乘坐城区公交车的通知...
27,E028,But when he was defeated by his counterpart Hu...,田亮参加了两次奥运会的单双人十米台比赛，并获得了2金2银。
28,E029,According to the recent report of the Ministry...,根据民政部最近发布的情况，全国6000多万人受灾，直接经济损失超过2000亿元。


In [26]:
negation_cases = candidate_cases[
    candidate_cases["has_negation"]
][
    [
        "eval_id",
        "source_en",
        "reference_zh"
    ]
]

negation_cases

,eval_id,source_en,reference_zh
12,E013,"As never before or since, home economists - am...",家庭经济学家们影响着这个国家的饮食习惯，比如1923年至1943年担任联邦家庭经济局主厨的路...
14,E015,The idiot didn't realize they were still on th...,这个白痴没有意识到是在直播。
30,E031,Zhang Xiaolong showed the animation show of he...,张小龙将女儿的动漫画作展示给记者，“如果没有当射击运动员，她应该还会继续画画”。
32,E033,"With the popularity of smart phones, mobile ph...",随着智能手机的普及，手机越来越好玩，刷微信、发微博乃至玩游戏、逛淘宝都成常态，很多人似乎须臾...
33,E034,Parker had a better performance than last seas...,帕克在上个赛季表现得比数据反映的要好些，不仅是因为他的篮下效率来到了生涯最高，他和阿尔德里奇...
37,E038,"Last year, officials said, Mr. Hooker's team c...",去年，有官员表示，胡克先生的团队所得出的结论是针对伊斯兰国炼油厂的空袭并未大幅削减恐怖组织的...
42,E043,"We are witnessing the top of the sport, but it...",我们登上了这项运动的巅峰，但在全美体操教学课堂上却并非如此。
46,E047,"""I thought my elder brother and I cannot affor...",她感激地说：“我原以为爸爸去世了，我和哥哥都上不成大学了，没想到县里及时为我们解决了困难。”


In [27]:
modality_cases = candidate_cases[
    candidate_cases["has_modality"]
][
    [
        "eval_id",
        "source_en",
        "reference_zh"
    ]
]

modality_cases

,eval_id,source_en,reference_zh
6,E007,"According to the navigation needs, the old Wan...","根据通航需要,老万福河航道采取养护改造的方式,对局部水深进行了浚深,局部急弯进行裁弯取直等,..."
7,E008,Even though she learns and works in decentrali...,即便通过分散的，模块化的时间来学习和工作，她也基本上都可以完成任务，并取得不错的成绩。
9,E010,According to the forecast of Ministry of Trade...,根据塔贸易和经济发展部的预测，2016年塔国的经济增长将达到7%，而国家的通胀也基本是在这一...
18,E019,"Their new, collaborative work, ""A Square Meal,...",他们共同编写的新书《丰盛大餐》中将在周二由哈珀出版社出版，这本书介绍的是大萧条时期美国的食品历史。
23,E024,"Prior to this, Jining Municipal Committee for ...",此前，济宁市老龄委、市财政局、市交通局下发《关于60—64周岁老年人半价乘坐城区公交车的通知...
25,E026,"We can bring those brands back up again, so I ...",我认为我们可以重新启动这些品牌，而且现在时间正合适。
29,E030,He had told BMA members the deal was a good on...,"在54,000名初级医生与医科学生投票之前的会议中，他曾对英国医学协会成员表示，该交易不错，..."
30,E031,Zhang Xiaolong showed the animation show of he...,张小龙将女儿的动漫画作展示给记者，“如果没有当射击运动员，她应该还会继续画画”。
38,E039,Whether it is because of the increase in age o...,无论是因为年龄的增大还是出场时间被限制，每个赛季都有一些球员的表现会出现退步。
47,E048,"""We hope that by comparing our results to othe...",帕萨迪纳加州理工学院红外处理分析中心研究科学家卢萨·雷布尔在新闻发布会上表示：“我们希望通过...


In [28]:
temporal_cases = candidate_cases[
    candidate_cases["has_temporal"]
][
    [
        "eval_id",
        "source_en",
        "reference_zh"
    ]
]

temporal_cases

,eval_id,source_en,reference_zh
0,E001,Democratic members of the House Intelligence C...,众议院情报委员会的民主党成员于周四公布了其调查结果，与共和党专项工作小组的部分结论相一致。
15,E016,This is despite a promise from Jeremy Hunt jus...,尽管如此，就在上个月我们还是得到了杰里米·亨特的承诺，他说自己的大门始终敞开。
18,E019,"Their new, collaborative work, ""A Square Meal,...",他们共同编写的新书《丰盛大餐》中将在周二由哈珀出版社出版，这本书介绍的是大萧条时期美国的食品历史。
20,E021,"As of June 2016, the world conducted a total o...",截至2016年6月，全球共进行了114次月球探测，43次火星探测。
22,E023,Great Britain's impressive run in the Rugby 7s...,在周四晚上的七人制橄榄球赛决赛中，英国队未能在斐济队面前延续之前的出色表现，被斐济队以43比...
23,E024,"Prior to this, Jining Municipal Committee for ...",此前，济宁市老龄委、市财政局、市交通局下发《关于60—64周岁老年人半价乘坐城区公交车的通知...
37,E038,"Last year, officials said, Mr. Hooker's team c...",去年，有官员表示，胡克先生的团队所得出的结论是针对伊斯兰国炼油厂的空袭并未大幅削减恐怖组织的...
40,E041,Three years ago he initially struggled when re...,三年前，当他在六个月禁期结束后重回赛场时，曾一度很吃力。但是，他表现出了惊人的性格优势，并终...
41,E042,The result is mean climate temperatures that a...,模拟的结果是金星的平均气候温度实际上比现在的地球还低了几度。
43,E044,"In July 2015, on Anzhen Street of Wuxi Xishan ...",2015年7月，无锡市锡山区安镇街道14个农村经济合作组织与所在村村委会“分账管理”，村委会...


In [29]:
controlled_cases = [
    {
        "case_id": "NUM_01",
        "eval_id": "E021",
        "category": "number",
        "perturbed_zh": "截至2016年6月，全球共进行了114次月球探测，34次火星探测。",
        "error_description": "Mars exploration count changed from 43 to 34."
    },
    {
        "case_id": "NUM_02",
        "eval_id": "E028",
        "category": "number",
        "perturbed_zh": "田亮参加了两次奥运会的单双人十米台比赛，并获得了1金2银。",
        "error_description": "Gold medal count changed from 2 to 1."
    },

    {
        "case_id": "NEG_01",
        "eval_id": "E015",
        "category": "negation",
        "perturbed_zh": "这个白痴意识到是在直播。",
        "error_description": "Negation removed: 'did not realize' becomes 'realized'."
    },
    {
        "case_id": "NEG_02",
        "eval_id": "E038",
        "category": "negation",
        "perturbed_zh": "去年，有官员表示，胡克先生的团队所得出的结论是针对伊斯兰国炼油厂的空袭大幅削减了恐怖组织的收入。",
        "error_description": "Negation removed, reversing whether the airstrikes substantially reduced revenue."
    },

    {
        "case_id": "TIME_01",
        "eval_id": "E001",
        "category": "temporal",
        "perturbed_zh": "众议院情报委员会的民主党成员于周五公布了其调查结果，与共和党专项工作小组的部分结论相一致。",
        "error_description": "Thursday changed to Friday."
    },
    {
        "case_id": "TIME_02",
        "eval_id": "E044",
        "category": "temporal",
        "perturbed_zh": "2016年7月，无锡市锡山区安镇街道14个农村经济合作组织与所在村村委会“分账管理”，村委会和合作社各有一本账。",
        "error_description": "Year changed from 2015 to 2016."
    },

    {
        "case_id": "MOD_01",
        "eval_id": "E026",
        "category": "modality",
        "perturbed_zh": "我认为我们必须重新启动这些品牌，而且现在时间正合适。",
        "error_description": "Ability ('can') changed to obligation ('must')."
    },
    {
        "case_id": "MOD_02",
        "eval_id": "E031",
        "category": "modality",
        "perturbed_zh": "张小龙将女儿的动漫画作展示给记者，“如果没有当射击运动员，她一定会继续画画”。",
        "error_description": "Possibility ('should/would likely') changed to certainty."
    }
]

controlled_df = pd.DataFrame(controlled_cases)

controlled_df = controlled_df.merge(
    eval_df[
        ["eval_id", "source_en", "reference_zh"]
    ],
    on="eval_id",
    how="left"
)

controlled_df = controlled_df.rename(
    columns={"reference_zh": "clean_zh"}
)

assert len(controlled_df) == 8
assert not controlled_df[
    ["source_en", "clean_zh", "perturbed_zh"]
].isna().any().any()

controlled_df[
    [
        "case_id",
        "category",
        "source_en",
        "clean_zh",
        "perturbed_zh",
        "error_description"
    ]
]

,case_id,category,source_en,clean_zh,perturbed_zh,error_description
0,NUM_01,number,"As of June 2016, the world conducted a total o...",截至2016年6月，全球共进行了114次月球探测，43次火星探测。,截至2016年6月，全球共进行了114次月球探测，34次火星探测。,Mars exploration count changed from 43 to 34.
1,NUM_02,number,But when he was defeated by his counterpart Hu...,田亮参加了两次奥运会的单双人十米台比赛，并获得了2金2银。,田亮参加了两次奥运会的单双人十米台比赛，并获得了1金2银。,Gold medal count changed from 2 to 1.
2,NEG_01,negation,The idiot didn't realize they were still on th...,这个白痴没有意识到是在直播。,这个白痴意识到是在直播。,Negation removed: 'did not realize' becomes 'r...
3,NEG_02,negation,"Last year, officials said, Mr. Hooker's team c...",去年，有官员表示，胡克先生的团队所得出的结论是针对伊斯兰国炼油厂的空袭并未大幅削减恐怖组织的...,去年，有官员表示，胡克先生的团队所得出的结论是针对伊斯兰国炼油厂的空袭大幅削减了恐怖组织的收入。,"Negation removed, reversing whether the airstr..."
4,TIME_01,temporal,Democratic members of the House Intelligence C...,众议院情报委员会的民主党成员于周四公布了其调查结果，与共和党专项工作小组的部分结论相一致。,众议院情报委员会的民主党成员于周五公布了其调查结果，与共和党专项工作小组的部分结论相一致。,Thursday changed to Friday.
5,TIME_02,temporal,"In July 2015, on Anzhen Street of Wuxi Xishan ...",2015年7月，无锡市锡山区安镇街道14个农村经济合作组织与所在村村委会“分账管理”，村委会...,2016年7月，无锡市锡山区安镇街道14个农村经济合作组织与所在村村委会“分账管理”，村委会...,Year changed from 2015 to 2016.
6,MOD_01,modality,"We can bring those brands back up again, so I ...",我认为我们可以重新启动这些品牌，而且现在时间正合适。,我认为我们必须重新启动这些品牌，而且现在时间正合适。,Ability ('can') changed to obligation ('must').
7,MOD_02,modality,Zhang Xiaolong showed the animation show of he...,张小龙将女儿的动漫画作展示给记者，“如果没有当射击运动员，她应该还会继续画画”。,张小龙将女儿的动漫画作展示给记者，“如果没有当射击运动员，她一定会继续画画”。,Possibility ('should/would likely') changed to...


In [30]:
FAILURE_JUDGE_PROMPT = """
You are evaluating the quality of an English-to-Mandarin Chinese translation.

English source:
{source}

Reference Mandarin translation:
{reference}

Candidate Mandarin translation:
{candidate}

Evaluate the candidate translation using the following criteria:

Accuracy:
Does the candidate correctly preserve the meaning of the English source?

Completeness:
Does the candidate preserve all important information without omissions
or unsupported additions?

Fluency:
Is the Mandarin grammatically correct, natural, and readable?

Terminology:
Are words, expressions, names, and domain-specific terms translated
appropriately?

Based on these criteria, give one overall score from 1 to 5 using the
following scoring anchors:

5 = Excellent
The meaning is fully preserved. There are no meaningful mistranslations,
omissions, or unsupported additions.

4 = Good
The meaning is substantially preserved. Minor issues may exist but do not
materially change the source meaning.

3 = Acceptable
The main meaning is preserved, but noticeable errors reduce translation quality.

2 = Poor
Important information is mistranslated, omitted, or incorrectly added.

1 = Very poor
The translation substantially fails to convey the source meaning.

Return your answer exactly in this format:

Score: <1-5>
Explanation: <brief explanation>
""".strip()

In [31]:
experiment_rows = []

for _, case in controlled_df.iterrows():

    for condition, candidate in [
        ("clean", case["clean_zh"]),
        ("perturbed", case["perturbed_zh"])
    ]:

        for run in range(1, 6):

            prompt = FAILURE_JUDGE_PROMPT.format(
                source=case["source_en"],
                reference=case["clean_zh"],
                candidate=candidate
            )

            experiment_rows.append({
                "case_id": case["case_id"],
                "eval_id": case["eval_id"],
                "category": case["category"],
                "condition": condition,
                "run": run,
                "source_en": case["source_en"],
                "reference_zh": case["clean_zh"],
                "candidate_zh": candidate,
                "error_description": case["error_description"],
                "prompt": prompt
            })

experiment_plan = pd.DataFrame(experiment_rows)

print("Cases:", controlled_df["case_id"].nunique())
print("Conditions:", experiment_plan["condition"].unique())
print("Runs per condition:", experiment_plan["run"].nunique())
print("Total planned judge calls:", len(experiment_plan))

assert len(experiment_plan) == 80

Cases: 8
Conditions: ['clean' 'perturbed']
Runs per condition: 5
Total planned judge calls: 80


In [32]:
print(
    experiment_plan.groupby(
        ["category", "condition"]
    ).size()
)

assert experiment_plan["prompt"].notna().all()
assert experiment_plan["candidate_zh"].notna().all()

print("\nExperiment ready for GPU.")

category  condition
modality  clean        10
          perturbed    10
negation  clean        10
          perturbed    10
number    clean        10
          perturbed    10
temporal  clean        10
          perturbed    10
dtype: int64

Experiment ready for GPU.


In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
from tqdm.auto import tqdm

JUDGE_MODEL = "Qwen/Qwen3-4B"

print("Loading judge:", JUDGE_MODEL)

judge_tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL
)

judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    torch_dtype="auto",
    device_map="auto"
)

judge_model.eval()

print("Judge loaded.")
print("Device:", judge_model.device)


def run_judge(
    prompt,
    temperature=0.7,
    top_p=0.95,
    max_new_tokens=128
):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = judge_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = judge_tokenizer(
        text,
        return_tensors="pt"
    ).to(judge_model.device)

    with torch.no_grad():

        output = judge_model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            pad_token_id=judge_tokenizer.eos_token_id
        )

    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    response = judge_tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return response.strip()


def parse_score(response):

    match = re.search(
        r"Score:\s*([1-5])",
        response,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return None


failure_results = []

for _, row in tqdm(
    experiment_plan.iterrows(),
    total=len(experiment_plan)
):

    response = run_judge(
        row["prompt"],
        temperature=0.7,
        top_p=0.95,
        max_new_tokens=128
    )

    score = parse_score(response)

    failure_results.append({
        "case_id": row["case_id"],
        "eval_id": row["eval_id"],
        "category": row["category"],
        "condition": row["condition"],
        "run": row["run"],
        "source_en": row["source_en"],
        "reference_zh": row["reference_zh"],
        "candidate_zh": row["candidate_zh"],
        "error_description": row["error_description"],
        "score": score,
        "valid_output": score is not None,
        "raw_response": response
    })


failure_results_df = pd.DataFrame(
    failure_results
)

failure_results_df.to_csv(
    "failure_case_results.csv",
    index=False
)

print("\nExperiment complete.")
print("Total evaluations:", len(failure_results_df))
print(
    "Valid outputs:",
    failure_results_df["valid_output"].sum()
)
print(
    "Invalid outputs:",
    (~failure_results_df["valid_output"]).sum()
)

Loading judge: Qwen/Qwen3-4B


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Judge loaded.
Device: cuda:0


  0%|          | 0/80 [00:00<?, ?it/s]


Experiment complete.
Total evaluations: 80
Valid outputs: 80
Invalid outputs: 0


In [34]:
# Mean score for each clean/perturbed condition
condition_scores = (
    failure_results_df
    .groupby(
        ["case_id", "category", "condition"]
    )["score"]
    .agg(
        mean_score="mean",
        std_score="std",
        min_score="min",
        max_score="max"
    )
    .reset_index()
)


comparison = (
    condition_scores
    .pivot(
        index=["case_id", "category"],
        columns="condition",
        values="mean_score"
    )
    .reset_index()
)

comparison["score_drop"] = (
    comparison["clean"]
    - comparison["perturbed"]
)

comparison["detected"] = (
    comparison["score_drop"] > 0
)

print("CONTROLLED FAILURE-CASE RESULTS")
print("=" * 70)

display(
    comparison.sort_values(
        "score_drop",
        ascending=False
    )
)


# Category-level summary
category_summary = (
    comparison
    .groupby("category")
    .agg(
        cases=("case_id", "count"),
        mean_score_drop=("score_drop", "mean"),
        detection_rate=("detected", "mean")
    )
    .reset_index()
)

category_summary["detection_percent"] = (
    category_summary["detection_rate"] * 100
)

print("\nCATEGORY SUMMARY")
display(category_summary)


print("\nOVERALL")
print(
    "Cases correctly distinguished:",
    comparison["detected"].sum(),
    "/",
    len(comparison)
)

print(
    "Overall detection rate:",
    round(
        comparison["detected"].mean() * 100,
        1
    ),
    "%"
)

CONTROLLED FAILURE-CASE RESULTS


condition,case_id,category,clean,perturbed,score_drop,detected
3,NEG_02,negation,5.0,3.0,2.0,True
2,NEG_01,negation,5.0,3.0,2.0,True
7,TIME_02,temporal,5.0,3.0,2.0,True
4,NUM_01,number,5.0,3.0,2.0,True
0,MOD_01,modality,5.0,4.0,1.0,True
1,MOD_02,modality,5.0,4.0,1.0,True
6,TIME_01,temporal,5.0,4.0,1.0,True
5,NUM_02,number,3.0,3.0,0.0,False



CATEGORY SUMMARY


,category,cases,mean_score_drop,detection_rate,detection_percent
0,modality,2,1.0,1.0,100.0
1,negation,2,2.0,1.0,100.0
2,number,2,1.0,0.5,50.0
3,temporal,2,1.5,1.0,100.0



OVERALL
Cases correctly distinguished: 7 / 8
Overall detection rate: 87.5 %
